# 평가 Gold Set 생성 개요

- **목적**

2만 개 stratified sample 도서 DB를 활용하여 retrieval/rerank 모델 실험용 평가셋을 생성한다.

현재 단계에서는 전체 사람 검수가 어렵기 때문에, LLM judge를 활용해 후보 도서에 대한 relevance pseudo-label을 생성하고, 이를 기반으로 1차 gold candidate set을 구성한다.

---

- **입력 데이터**

 1. `scenario_data.json`

- 경로: `evaluation/dataset/scenario_data.json`
- 구성:
  - `onboarding_data`: 사용자 장기 선호 정보
  - `session_data`: 현재 대화 세션에서 추출된 사용자 요청 정보

 2. 도서 DB

- 약 2만 개 stratified sample 도서 데이터
- BM25 / Dense / Hybrid 검색 대상
- ISBN 기준 도서 메타데이터 조회 대상

---

- **생성 과정**

1. `session_data`와 `onboarding_data`를 기반으로 검색 질의를 구성한다.

2. 도서 DB를 대상으로 BM25, Dense, Hybrid retriever를 각각 실행하여 Top-40 후보를 추출한다.

3. 세 retriever 결과를 merge하고, ISBN 기준으로 중복 제거한다.

4. 중복 제거된 ISBN을 기준으로 DB에서 도서 메타데이터를 조회한다.

5. LLM judge가 각 후보 도서의 relevance를 `0~3 grade`로 판단한다.

   - `session_data`를 우선 기준으로 사용한다.
   - `use_onboarding=true`인 경우 `onboarding_data`를 보조 기준으로 반영한다.
   - `session_data`와 `onboarding_data`가 충돌하면 `session_data`를 우선한다.
   - retrieval rank, score, source는 LLM judge 입력에서 제외한다.
   - 단, `retrieval_info`는 분석 및 hard negative 추출을 위해 출력 데이터에는 저장한다.

6. `relevance_grade`를 기반으로 label을 생성한다.

```python
binary_label = 1 if relevance_grade >= 2 else 0
```

출력 항목 예시:

- `relevance_grade`: 0, 1, 2, 3
- `binary_label`: 0 또는 1
- `retrieval_info`: source/rank/score 정보


In [5]:
import os
os.chdir("/home/ubuntu/work/somin/backend")
os.getcwd()

'/home/ubuntu/work/somin/backend'

In [4]:
import asyncio
import json
import uuid
import time
import random
import requests
import nest_asyncio
from pathlib import Path
from collections import defaultdict
from tqdm.auto import tqdm
from app.modules.RAG.retriever import (
    full_bm25_with_onboarding,
    full_dense_with_onboarding,
    full_hybrid_with_onboarding,
)
from app.core.config import settings
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

nest_asyncio.apply()

ModuleNotFoundError: No module named 'app.modules.RAG.retriever'

# Retriever 후보 Pool

In [ ]:
CLOVA_API_KEY = settings.CLOVA_API_KEY

REPO_ROOT     = Path("/Users/shimsomin/Workplace/git_workspace/librAIan")
DATASET_DIR   = REPO_ROOT / "evaluation" / "dataset"
SCENARIO_PATH = DATASET_DIR / "scenario_data.json"

OUTPUT_JSONL  = DATASET_DIR / "goldset_all_scenarios.jsonl"
OUTPUT_JSON   = DATASET_DIR / "goldset_all_scenarios.json"

URL         = "https://clovastudio.stream.ntruss.com/v3/chat-completions/HCX-007"
CONCURRENCY = 8    # 동시 요청 수 (429 발생 시 5로 낮춤)
MAX_RETRIES = 5
LIMIT       = None  # 테스트 시 숫자로 변경 (예: 10), 전체 실행 시 None

with open(SCENARIO_PATH, encoding="utf-8") as f:
    scenarios = json.load(f)

print(f"시나리오 수: {len(scenarios)}")
print(f"출력 JSONL: {OUTPUT_JSONL}")
print(f"출력 JSON: {OUTPUT_JSON}")
print(f"동시 요청 수: {CONCURRENCY}")
print(f"LIMIT: {LIMIT}")


def extract_rag_query(scenario: dict) -> dict:
    """마지막 turn의 rag_query를 꺼내고 query_id(=scenario_id)를 추가한다."""
    rag_query = scenario["turns"][-1]["rag_query"].copy()
    rag_query["query_id"] = scenario["scenario_id"]
    return rag_query

In [ ]:
def simplify_item(item, source_name):
    return {
        "isbn":         item.get("isbn"),
        "title":        item.get("title"),
        "author":       item.get("author"),
        "publisher":    item.get("publisher"),
        "publish_date": item.get("publish_date"),
        "page":         item.get("page"),
        "large_cate":   item.get("large_cate"),
        "mid_cate":     item.get("mid_cate"),
        "small_cate":   item.get("small_cate"),
        "book_intro":   item.get("book_intro"),
        "book_index":   item.get("book_index"),
        "review_count": item.get("review_count"),
        "review_text":  item.get("review_text"),
    }


def merge_with_sources(result_groups, key="isbn"):
    """여러 retriever 결과를 isbn 기준으로 merge하고 retrieval_info를 기록한다."""
    merged = {}

    for source_name, results in result_groups.items():
        for item in results:
            item_key = item.get(key)
            if item_key is None:
                continue

            if item_key not in merged:
                merged[item_key] = simplify_item(item, source_name)
                merged[item_key]["retrieval_info"] = []

            merged[item_key]["retrieval_info"].append({
                "source": source_name,
                "rank":   item.get("rank"),
                "score":  item.get("score"),
            })

    return list(merged.values())

# LLM-Judge labeling

In [ ]:
SYSTEM_PROMPT = """
당신은 매우 보수적으로 책 추천 적합성을 판단하는 평가자입니다.

입력으로는 다음 두 정보가 주어집니다.

1. request: 사용자의 검색 요청 전체
- keyword_query
- semantic_query
- filters
- constraints
- score_boost
- session_signals
- onboarding_signals

2. book: 추천 후보 도서 정보

[판정 기준]
- 현재 사용자 요청의 핵심 의도와 조건을 우선적으로 판단하세요.
- session_signals를 최우선 기준으로 판단하세요.
- onboarding_signals는 session_signals와 충돌하지 않는 경우에만 보조적으로 반영하세요.
- session_signals와 onboarding_signals가 충돌하면 session_signals를 우선하세요.
- 사용자 요청의 모든 표현이 책 정보에 직접 등장할 필요는 없습니다.
- 제목, 소개, 목차, 카테고리, 리뷰 등을 종합하여 의미적 적합성을 판단하세요.
- 책이 제공하는 독서 경험, 분위기, 주제, 대상 독자가 사용자 요청과 충분히 일치하면 높은 점수를 부여하세요.
- 일부 키워드만 우연히 일치하거나, 장르/주제/대상/분위기가 실제로 맞지 않으면 낮은 점수를 부여하세요.
- filters, constraints는 명시 조건으로 보고 우선 고려하세요.
- soft 조건은 약간 벗어나도 핵심 요청 적합성이 높으면 높은 점수를 부여할 수 있습니다.
- 정보가 부족하여 핵심 조건 충족 여부를 판단할 수 없으면 낮은 점수를 부여하세요.
- 요청과 직접 관련 없는 책이 단순히 키워드만 포함한 경우에는 낮은 점수를 부여하세요.
- retrieval rank, score, source 정보는 판단에 사용하지 마세요.
- 반드시 JSON만 출력하세요. 설명 문장, markdown, 코드블록은 출력하지 마세요.

[Grade 기준]
3 = 매우 적합. 현재 요청의 핵심 의도와 조건에 잘 부합함.
2 = 적합. 핵심 의도는 맞지만 일부 조건이 약함.
1 = 부분 관련. 관련성은 있으나 정답 도서로 보기에는 약함.
0 = 부적합. 현재 요청의 핵심 의도와 맞지 않음.

출력 형식:
{
  "relevance_grade": 0 또는 1 또는 2 또는 3,
  "reason": "판정 근거를 간략히 설명하는 텍스트"
}
"""

EXCLUDE_KEYS = {
    "retrieval_info",
    "rank", "score", "source",
    "retrieval_source", "retrieval_rank", "retrieval_score",
    "bm25_score", "dense_score", "hybrid_score",
    "main_score", "onboarding_score",
    "disliked_penalty", "disliked_similarity",
    # 기존 평가 필드: 재실행 시 LLM 편향 방지
    "relevance_grade", "binary_label", "label_status",
    "llm_raw_output", "llm_error", "query_id",
}


def make_headers():
    return {
        "Authorization": f"Bearer {CLOVA_API_KEY}",
        "X-NCP-CLOVASTUDIO-REQUEST-ID": str(uuid.uuid4()),
        "Content-Type": "application/json",
        "Accept": "text/event-stream",
    }


def make_book_for_judge(book):
    """LLM judge 입력에서 retrieval 정보 및 기존 평가 필드를 제거한다."""
    return {
        k: v for k, v in book.items()
        if k not in EXCLUDE_KEYS and not str(k).startswith("retrieval_")
    }


def parse_hcx_response(text):
    """SSE 형식 또는 일반 JSON 형식 응답을 모두 처리한다."""
    if "event:" in text:
        last_data = None
        for block in text.split("\n\n"):
            lines = block.strip().splitlines()
            event_type = data_text = None
            for line in lines:
                if line.startswith("event:"):
                    event_type = line.replace("event:", "").strip()
                elif line.startswith("data:"):
                    data_text = line.replace("data:", "").strip()
            if event_type == "result" and data_text:
                last_data = data_text
        if last_data is not None:
            data = json.loads(last_data)
            return data["message"]["content"]

    try:
        data = json.loads(text)
        if "result" in data:
            return data["result"]["message"]["content"]
        if "message" in data:
            return data["message"]["content"]
    except Exception:
        pass

    raise ValueError(f"응답 파싱 실패:\n{text[:1000]}")


def extract_grade(llm_text):
    """파싱 실패 시 relevance_grade=None, label_status='parse_failed'로 반환한다."""
    try:
        obj = json.loads(llm_text)
        grade = obj.get("relevance_grade", obj.get("grade", obj.get("label")))
        if grade is None:
            return None, "parse_failed", "missing relevance_grade"
        grade = int(grade)
        if grade not in [0, 1, 2, 3]:
            return None, "parse_failed", f"invalid relevance_grade: {grade}"
        return grade, "success", None
    except Exception as e:
        return None, "parse_failed", str(e)


def grade_to_binary(grade):
    if grade is None:
        return None
    return 1 if grade >= 2 else 0


def make_user_prompt(request_data, book):
    payload = {
        "request": request_data,
        "book": make_book_for_judge(book),
    }
    return json.dumps(payload, ensure_ascii=False, indent=2)


def blocking_label_one_book(request_data, book, max_retries=MAX_RETRIES):
    """동기 HTTP 요청 함수. asyncio.to_thread()로 thread pool에서 호출된다."""
    query_id    = request_data.get("query_id", "unknown")
    user_prompt = make_user_prompt(request_data, book)

    body = {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_prompt},
        ],
        "topP": 0.1, "topK": 0, "max_tokens": 100,
        "temperature": 0.0, "repetitionPenalty": 1.0,
        "includeAiFilters": False, "seed": 42,
    }

    last_error = None

    for attempt in range(max_retries):
        try:
            res = requests.post(URL, headers=make_headers(), json=body, timeout=60)

            if res.status_code == 200:
                llm_text     = parse_hcx_response(res.text)
                grade, label_status, parse_error = extract_grade(llm_text)
                binary_label = grade_to_binary(grade)
                return {
                    **book,
                    "query_id":        query_id,
                    "relevance_grade": grade,
                    "binary_label":    binary_label,
                    "label_status":    label_status,
                    "llm_raw_output":  llm_text,
                    "llm_error":       parse_error,
                }

            last_error = f"{res.status_code} / {res.text[:500]}"

            if res.status_code in [429, 500, 502, 503, 504]:
                raise Exception(last_error)

            return {
                **book,
                "query_id":        query_id,
                "relevance_grade": None,
                "binary_label":    None,
                "label_status":    "api_failed",
                "llm_raw_output":  None,
                "llm_error":       last_error,
            }

        except Exception as e:
            last_error = str(e)
            if attempt == max_retries - 1:
                return {
                    **book,
                    "query_id":        query_id,
                    "relevance_grade": None,
                    "binary_label":    None,
                    "label_status":    "api_failed",
                    "llm_raw_output":  None,
                    "llm_error":       last_error,
                }
            wait = min(30, (2 ** attempt) + random.uniform(0, 1))
            time.sleep(wait)


async def label_one_book_async(request_data, book, semaphore, write_lock, jsonl_file, pbar):
    """
    비동기 래퍼:
    - semaphore로 동시 요청 수 제한
    - asyncio.to_thread로 blocking HTTP 호출을 thread pool에서 실행
    - write_lock으로 JSONL 파일 쓰기 직렬화
    """
    async with semaphore:
        try:
            result = await asyncio.to_thread(
                blocking_label_one_book,
                request_data,
                book,
            )
        except Exception as e:
            query_id = request_data.get("query_id", "unknown")
            result = {
                **book,
                "query_id":        query_id,
                "relevance_grade": None,
                "binary_label":    None,
                "label_status":    "api_failed",
                "llm_raw_output":  None,
                "llm_error":       str(e),
            }

    async with write_lock:
        jsonl_file.write(json.dumps(result, ensure_ascii=False) + "\n")
        jsonl_file.flush()

    pbar.update(1)
    return result

In [ ]:
# ============================================================
# Phase 1: 시나리오별 Retrieval (순차 실행)
# ============================================================
# retrieval은 내부적으로 ES 등 동기 클라이언트를 쓰므로 순차 처리한다.
# 완료 후 all_candidates에 전체 후보가 (query_id 부착 상태로) 쌓인다.

all_candidates: list = []

for scenario in tqdm(scenarios, desc="시나리오 retrieval"):
    sample_result = extract_rag_query(scenario)
    if sample_result.get("anchor"):
            sample_result = run_anchor_pipeline(sample_result)
            
    query_id      = sample_result["query_id"]

    re_bm25   = full_bm25_with_onboarding(sample_result, size=40)
    re_dense  = full_dense_with_onboarding(sample_result, size=40)
    re_hybrid = full_hybrid_with_onboarding(sample_result, size=40)

    full_books = merge_with_sources({
        "bm25":   re_bm25,
        "dense":  re_dense,
        "hybrid": re_hybrid,
    })

    for book in full_books:
        book["query_id"] = query_id
        all_candidates.append(book)

print(f"\n전체 후보 도서: {len(all_candidates):,}")

# Phase 2: LLM 라벨링 (비동기)

> **테스트**: Cell 4의 `LIMIT = 10` 으로 설정 후 실행  
> **전체**: `LIMIT = None` 으로 설정 후 실행

In [ ]:
async def main():
    # query_map: scenario_id → rag_query
    query_map = {}
    for scenario in scenarios:
        rag_query = scenario["turns"][-1]["rag_query"].copy()
        rag_query["query_id"] = scenario["scenario_id"]
        query_map[scenario["scenario_id"]] = rag_query

    # resume: 이미 처리된 (query_id, isbn) 스킵
    processed_keys: set = set()
    already_labeled: list = []

    if OUTPUT_JSONL.exists():
        with OUTPUT_JSONL.open("r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                try:
                    item = json.loads(line)
                    key  = (item.get("query_id"), item.get("isbn"))
                    if key not in processed_keys:
                        processed_keys.add(key)
                        already_labeled.append(item)
                except json.JSONDecodeError:
                    continue
        print(f"이미 처리된 도서 수: {len(processed_keys):,}")

    # 처리 대상 필터링
    to_process = [
        book for book in all_candidates
        if (book.get("query_id"), book.get("isbn")) not in processed_keys
        and book.get("query_id") in query_map
    ]

    if LIMIT is not None:
        to_process = to_process[:LIMIT]

    print(f"전체 후보: {len(all_candidates):,}  |  처리 대상: {len(to_process):,}")

    if not to_process:
        print("처리할 항목이 없습니다.")
        return already_labeled

    # 비동기 실행
    semaphore  = asyncio.Semaphore(CONCURRENCY)
    write_lock = asyncio.Lock()
    pbar       = tqdm(total=len(to_process), desc="라벨링 중", unit="book")

    with OUTPUT_JSONL.open("a", encoding="utf-8") as jsonl_file:
        tasks = [
            label_one_book_async(
                query_map[book["query_id"]],
                book,
                semaphore,
                write_lock,
                jsonl_file,
                pbar,
            )
            for book in to_process
        ]
        new_results = await asyncio.gather(*tasks)

    pbar.close()

    # 원본 순서 기준 정렬 후 JSON 저장
    all_results = already_labeled + list(new_results)
    order = {(e.get("query_id"), e.get("isbn")): i for i, e in enumerate(all_candidates)}
    all_results.sort(key=lambda x: order.get((x.get("query_id"), x.get("isbn")), 99999))

    with OUTPUT_JSON.open("w", encoding="utf-8") as f:
        json.dump(all_results, f, ensure_ascii=False, indent=2)

    print(f"\nJSONL 저장 완료: {OUTPUT_JSONL}")
    print(f"JSON  저장 완료: {OUTPUT_JSON}")

    # 통계
    stats      = defaultdict(lambda: defaultdict(int))
    fail_stats = defaultdict(int)
    for item in all_results:
        qid    = item.get("query_id", "unknown")
        grade  = item.get("relevance_grade")
        status = item.get("label_status")
        if status == "success":
            stats[qid][grade] += 1
        else:
            fail_stats[qid] += 1

    total_success = sum(sum(g.values()) for g in stats.values())
    total_failed  = sum(fail_stats.values())
    print(f"\n총 항목: {len(all_results):,}  성공: {total_success:,}  실패: {total_failed:,}")

    return all_results


results = await main()

# api_failed 재시도

`main()` 완료 후 `label_status == "api_failed"` 항목만 골라 재호출한다.  
성공한 항목은 그대로 유지하고, 결과는 OUTPUT_JSON에 덮어쓴다.

In [ ]:
async def retry_api_failed():
    """api_failed + parse_failed 항목을 OUTPUT_JSON에서 읽어 재호출 후 덮어쓴다."""
    if not OUTPUT_JSON.exists():
        print(f"파일 없음: {OUTPUT_JSON}\nmain()을 먼저 실행하세요.")
        return []

    with OUTPUT_JSON.open("r", encoding="utf-8") as f:
        all_results = json.load(f)

    # query_map 재구성
    query_map = {}
    for scenario in scenarios:
        rag_query = scenario["turns"][-1]["rag_query"].copy()
        rag_query["query_id"] = scenario["scenario_id"]
        query_map[scenario["scenario_id"]] = rag_query

    to_retry = [
        e for e in all_results
        if e.get("label_status") in ("api_failed", "parse_failed")
        or e.get("relevance_grade") is None
    ]
    print(f"전체: {len(all_results):,}개  |  재시도 대상: {len(to_retry):,}개")

    if not to_retry:
        print("재시도할 항목이 없습니다.")
        return all_results

    semaphore = asyncio.Semaphore(CONCURRENCY)
    pbar      = tqdm(total=len(to_retry), desc="재시도 중", unit="book")

    async def retry_one(entry):
        query_id = entry.get("query_id", "unknown")
        if query_id not in query_map:
            pbar.update(1)
            return entry
        async with semaphore:
            try:
                result = await asyncio.to_thread(
                    blocking_label_one_book,
                    query_map[query_id],
                    entry,
                )
            except Exception as e:
                result = {**entry, "label_status": "api_failed", "llm_error": str(e)}
        pbar.update(1)
        return result

    new_results = await asyncio.gather(*[retry_one(e) for e in to_retry])
    pbar.close()

    key_to_new = {(r.get("query_id"), r.get("isbn")): r for r in new_results}
    updated = [
        key_to_new.get((e.get("query_id"), e.get("isbn")), e)
        for e in all_results
    ]

    with OUTPUT_JSON.open("w", encoding="utf-8") as f:
        json.dump(updated, f, ensure_ascii=False, indent=2)

    now_success  = sum(1 for r in new_results if r.get("label_status") == "success")
    still_failed = sum(1 for r in new_results if r.get("label_status") in ("api_failed", "parse_failed"))
    print(f"\n재시도 결과 ({len(to_retry)}개)")
    print(f"  성공:       {now_success:,}")
    print(f"  여전히 실패:  {still_failed:,}")
    print(f"\nJSON 저장 완료: {OUTPUT_JSON}")
    return updated


results = await retry_api_failed()

# 결과 확인

In [ ]:
# main() 또는 retry_api_failed() 결과를 DataFrame으로 변환
# OUTPUT_JSON에서 다시 로드하려면 아래 주석을 해제한다.
# with OUTPUT_JSON.open("r", encoding="utf-8") as f:
#     results = json.load(f)

df = pd.DataFrame(results)

print(f"전체 라벨링 결과 수: {len(df):,}")
df.head()

In [8]:
status_dist = df["label_status"].value_counts(dropna=False).reset_index()
status_dist.columns = ["label_status", "count"]
status_dist["ratio"] = status_dist["count"] / len(df)

status_dist

,label_status,count,ratio
0,success,86,1.0


In [9]:
success_df = df[df["label_status"] == "success"].copy()

print(f"성공 라벨 수: {len(success_df):,}")
print(f"실패/제외 수: {len(df) - len(success_df):,}")

성공 라벨 수: 86
실패/제외 수: 0


In [10]:
grade_dist = (
    success_df["relevance_grade"]
    .value_counts()
    .sort_index()
    .reset_index()
)

grade_dist.columns = ["relevance_grade", "count"]
grade_dist["ratio"] = grade_dist["count"] / len(success_df)

grade_dist

,relevance_grade,count,ratio
0,0,7,0.081395
1,1,61,0.709302
2,2,15,0.174419
3,3,3,0.034884
